# ARC26 global-120 LoRA + local-80 validation

Controlled evaluation on the established 48 tasks against the 31.94 Vanilla V2
stack. A separately trained rank-256 global LoRA covering all 120 evaluation
tasks is merged into the untouched published model, then each evaluated puzzle
receives ordinary 80-step local TTFT. Candidate generation and selection are
unchanged: q_len=9, threshold 0.2, eight geometries by three colour/order
variants, and `score_kgmon`. No Repair or scheduled sampling is used.


In [ ]:
import os

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["OMP_NUM_THREADS"] = "12"


In [ ]:
MODE = "validation"

CODE_DATASET_ROOT = "/kaggle/input/datasets/yuvraj/arc2026"
BASE_MODEL_PATH = "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
GLOBAL_RUN_ROOT = "/kaggle/input/notebooks/yuvraj/arc26-global120-vanilla-lora/global120_vanilla_lora"
GLOBAL_ADAPTER = GLOBAL_RUN_ROOT + "/adapter"
MODEL_PATH = "/kaggle/working/global120_vanilla_merged_model"
COMP_ROOT = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"

VALIDATION_KEYS = ['0934a4d8', '135a2760', '136b0064', '13e47133', '142ca369', '16b78196', '16de56c4', '1818057f', '195c6913', '1ae2feb7', '20270e3b', '20a9e565', '21897d95', '221dfab4', '247ef758', '269e22fb', '271d71e2', '28a6681f', '291dc1e1', '2b83f449', '2ba387bc', '2c181942', '2d0172a1', '31f7f899', '332f06d7', '35ab12c3', '36a08778', '38007db0', '3a25b0d8', '3dc255db', '3e6067c3', '409aa875', '446ef5d2', '45a5af55', '4a21e3da', '4c3d4a41', '4c416de3', '4c7dc4dd', '4e34c42c', '53fb4810', '5545f144', '581f7754', '58490d8a', '58f5dbd5', '5961cc34', '5dbc8537', '62593bfd', '64efde09']
NPROCS = 4
LOCAL_TTFT_STEPS = 80
DFS_PROB_THRESHOLD = 0.2
UNSLOTH_MULTITOKEN_REPEAT_LEN = 9
EVAL_COLOR_PERMUTATIONS = 3
SELECTION_ALGORITHM = "score_kgmon"
PROFILE_TIMINGS = True

VALIDATION_END_TIME_HOURS = 4.0
SUBMIT_COMPETITION_END_TIME_HOURS = 11 + 50 / 60
RESET_RUN_ARTIFACTS = True

WORK_NOTEBOOK_ROOT = "/kaggle/working/arc2026_run_global120_local80_q9_24"
WORK_CODE_DIR = "/kaggle/working/arc2026_run_global120_local80_q9_24/ARC-AGI1/qwen_baseline"
WRITABLE_UNSLOTH_PARENT = "/kaggle/working/global120_local80_q9_24_stack"


In [ ]:
import os
from pathlib import Path


def _truthy_env(name: str) -> bool:
    value = os.getenv(name)
    if value is None:
        return False
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


IS_KAGGLE_RERUN = _truthy_env("KAGGLE_IS_COMPETITION_RERUN")
assert MODE in {"validation", "submit_competition"}
EFFECTIVE_MODE = "submit_competition" if IS_KAGGLE_RERUN else MODE

EVAL_CHALLENGES = f"{COMP_ROOT}/arc-agi_evaluation_challenges.json"
EVAL_SOLUTIONS = f"{COMP_ROOT}/arc-agi_evaluation_solutions.json"
TEST_CHALLENGES = f"{COMP_ROOT}/arc-agi_test_challenges.json"

if IS_KAGGLE_RERUN:
    TEST_PATH = TEST_CHALLENGES
    SOLUTION_PATH = None
    OUTPUT_DIR = "/kaggle/working/inference_outputs_vanilla_v2_q9_24_submit"
    SUBMISSION_PATH = "/kaggle/working/submission.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = None
    END_TIME_HOURS = SUBMIT_COMPETITION_END_TIME_HOURS
    RUN_INFERENCE = True
elif MODE == "validation":
    TEST_PATH = EVAL_CHALLENGES
    SOLUTION_PATH = EVAL_SOLUTIONS
    OUTPUT_DIR = "/kaggle/working/inference_outputs_vanilla_v2_q9_24_validation"
    SUBMISSION_PATH = "/kaggle/working/validation_submission_unsloth_q9.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = VALIDATION_KEYS
    END_TIME_HOURS = VALIDATION_END_TIME_HOURS
    RUN_INFERENCE = True
else:
    TEST_PATH = TEST_CHALLENGES
    SOLUTION_PATH = None
    OUTPUT_DIR = "/kaggle/working/inference_outputs_vanilla_v2_q9_24_shortcut"
    SUBMISSION_PATH = "/kaggle/working/submission.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = None
    END_TIME_HOURS = 0.0
    RUN_INFERENCE = False

print("mode_requested =", MODE)
print("is_kaggle_rerun =", IS_KAGGLE_RERUN)
print("effective_mode =", EFFECTIVE_MODE)
print("test_path =", TEST_PATH)
print("output_dir =", OUTPUT_DIR)
print("submission_path =", SUBMISSION_PATH)
print("selected_keys =", SELECTED_KEYS)
print("end_time_hours =", END_TIME_HOURS)
print("run_inference =", RUN_INFERENCE)


In [ ]:
import importlib.util
import os
import shutil
import sys
from pathlib import Path

assert Path(CODE_DATASET_ROOT).exists(), f"Missing code dataset root: {CODE_DATASET_ROOT}"
assert Path(BASE_MODEL_PATH).exists(), f"Missing base model path: {BASE_MODEL_PATH}"
assert Path(GLOBAL_ADAPTER).exists(), f"Missing global adapter: {GLOBAL_ADAPTER}"
assert Path(TEST_PATH).exists(), f"Missing challenge path: {TEST_PATH}"
assert Path(os.environ["TRITON_PTXAS_PATH"]).exists(), os.environ["TRITON_PTXAS_PATH"]
if SOLUTION_PATH is not None:
    assert Path(SOLUTION_PATH).exists(), f"Missing solution path: {SOLUTION_PATH}"

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"

os.chdir("/kaggle/working")
print("setup cwd =", os.getcwd())

if RESET_RUN_ARTIFACTS:
    for path in [WORK_NOTEBOOK_ROOT, OUTPUT_DIR, WRITABLE_UNSLOTH_PARENT, MODEL_PATH, "/kaggle/working/global120_adapter_compat"]:
        shutil.rmtree(path, ignore_errors=True)
    try:
        Path(SUBMISSION_PATH).unlink()
    except FileNotFoundError:
        pass
    for path in Path("/kaggle/working").glob("worker_train_*"):
        if path.is_file():
            path.unlink()

for module_name in ["unsloth", "transformers", "torch"]:
    spec = importlib.util.find_spec(module_name)
    print(module_name, spec.origin if spec else "MISSING")


In [ ]:
import os
import shutil
from pathlib import Path

src = Path(CODE_DATASET_ROOT)
dst = Path(WORK_NOTEBOOK_ROOT)
shutil.copytree(src, dst)

required_files = [
    "starter.py",
    "arc_solver.py",
    "arc_search_multitoken.py",
    "patch_unsloth_qwen3_multitoken.py",
]
for name in required_files:
    assert Path(WORK_CODE_DIR, name).is_file(), (
        f"arc2026 is stale: missing {name}"
    )

starter_source = Path(WORK_CODE_DIR, "starter.py").read_text()
solver_source = Path(WORK_CODE_DIR, "arc_solver.py").read_text()
assert "--use-unsloth-multitoken-dfs" in starter_source
assert "--eval-color-permutations" in starter_source
assert "UNSLOTH_COMPILE_LOCATION" in starter_source, "arc2026 starter.py lacks the worker import-race guard"
assert '"embed_tokens"' in solver_source and '"lm_head"' in solver_source
assert "inference_turbo_dfs_multitoken" in solver_source
print("arc2026 multi-token production preflight passed")
print("work_code_dir =", WORK_CODE_DIR)


In [ ]:
import subprocess
import textwrap
import time

merge_code = textwrap.dedent(r"""
import json
import os
import shutil
from pathlib import Path

import unsloth
from peft import PeftModel
from unsloth import FastLanguageModel

base_path = os.environ["ARC_BASE_MODEL_PATH"]
adapter_path = Path(os.environ["ARC_GLOBAL_ADAPTER_PATH"])
compat_path = Path("/kaggle/working/global120_adapter_compat")
output_path = Path(os.environ["ARC_MERGED_MODEL_PATH"])

# Modern PEFT writes extra config fields which the pinned Vanilla V2 PEFT does
# not recognize. The weights and all training-relevant fields are unchanged.
supported = {
    "base_model_name_or_path", "bias", "fan_in_fan_out", "inference_mode",
    "init_lora_weights", "layers_pattern", "layers_to_transform", "loftq_config",
    "lora_alpha", "lora_dropout", "megatron_config", "megatron_core",
    "modules_to_save", "peft_type", "r", "rank_pattern", "revision",
    "target_modules", "task_type", "use_dora", "use_rslora",
}
raw = json.loads((adapter_path / "adapter_config.json").read_text())
config = {key: value for key, value in raw.items() if key in supported}
config["base_model_name_or_path"] = base_path
compat_path.mkdir(parents=True)
(compat_path / "adapter_config.json").write_text(json.dumps(config, indent=2) + "\n")
os.symlink(adapter_path / "adapter_model.safetensors", compat_path / "adapter_model.safetensors")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_path,
    full_finetuning=False,
    load_in_4bit=False,
    local_files_only=True,
    use_gradient_checkpointing=False,
    max_seq_length=8192,
)
model = PeftModel.from_pretrained(
    model, str(compat_path), is_trainable=False, local_files_only=True
)
model = model.merge_and_unload(safe_merge=True)
output_path.mkdir(parents=True)
model.save_pretrained(output_path, safe_serialization=True, max_shard_size="5GB")
tokenizer.save_pretrained(output_path)
print("saved global-merged model:", output_path)
""")

merge_env = os.environ.copy()
merge_env.update({
    "ARC_BASE_MODEL_PATH": BASE_MODEL_PATH,
    "ARC_GLOBAL_ADAPTER_PATH": GLOBAL_ADAPTER,
    "ARC_MERGED_MODEL_PATH": MODEL_PATH,
})
merge_started = time.perf_counter()
subprocess.run([sys.executable, "-c", merge_code], env=merge_env, check=True)
MERGE_WALL_SECONDS = time.perf_counter() - merge_started
print("merge_wall_s =", round(MERGE_WALL_SECONDS, 3))


In [ ]:
spec = importlib.util.find_spec("unsloth")
assert spec is not None and spec.submodule_search_locations
mounted_unsloth = Path(next(iter(spec.submodule_search_locations)))
qwen_source = (mounted_unsloth / "models" / "qwen3.py").read_text()
assert "A = flash_attn_func(Qnn, Knn, Vnn)" in qwen_source

writable_parent = Path(WRITABLE_UNSLOTH_PARENT)
writable_unsloth = writable_parent / "unsloth"
shutil.copytree(mounted_unsloth, writable_unsloth)

sys.path.insert(0, WORK_CODE_DIR)
from patch_unsloth_qwen3_multitoken import PATCH_MARKER, patch_unsloth

changed = patch_unsloth(writable_unsloth)
assert PATCH_MARKER in (writable_unsloth / "models" / "qwen3.py").read_text()
print("writable_unsloth =", writable_unsloth)
print("patched =", [str(path) for path in changed])

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = str(writable_parent) + os.pathsep + RUN_ENV.get("PYTHONPATH", "")


In [ ]:
import json
import os
import subprocess
import sys
import time

if RUN_INFERENCE:
    cmd = [
        sys.executable,
        "starter.py",
        "--test-path", TEST_PATH,
        "--model-path", MODEL_PATH,
        "--output-dir", OUTPUT_DIR,
        "--nprocs", str(NPROCS),
        "--full-sft-total-steps", str(LOCAL_TTFT_STEPS),
        "--use-unsloth-multitoken-dfs",
        "--unsloth-multitoken-repeat-len", str(UNSLOTH_MULTITOKEN_REPEAT_LEN),
        "--dfs-prob-threshold", str(DFS_PROB_THRESHOLD),
        "--eval-color-permutations", str(EVAL_COLOR_PERMUTATIONS),
        "--end-time", str(time.time() + END_TIME_HOURS * 3600),
    ]
    if PROFILE_TIMINGS:
        cmd.append("--profile-timings")
    if SELECTED_KEYS is not None:
        cmd.extend(["--keys-json", json.dumps(SELECTED_KEYS)])

    print("running:", " ".join(cmd), flush=True)
    inference_started = time.perf_counter()
    subprocess.run(cmd, cwd=WORK_CODE_DIR, env=RUN_ENV, check=True)
    INFERENCE_WALL_SECONDS = time.perf_counter() - inference_started
    print("inference_wall_s =", round(INFERENCE_WALL_SECONDS, 3))
    global_runtime = json.loads(Path(GLOBAL_RUN_ROOT, "global_notebook_runtime.json").read_text())
    print("merge_plus_local_wall_s =", round(MERGE_WALL_SECONDS + INFERENCE_WALL_SECONDS, 3))
    print("global_plus_merge_plus_local_wall_s =", round(
        global_runtime["global_notebook_wall_s"] + MERGE_WALL_SECONDS + INFERENCE_WALL_SECONDS,
        3,
    ))
else:
    print("save-version shortcut: full inference runs only during the competition rerun")


In [ ]:
import json
import os
import sys
from pathlib import Path

if WORK_CODE_DIR not in sys.path:
    sys.path.insert(0, WORK_CODE_DIR)

from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, score_kgmon, score_full_probmul_3

SELECTION_ALGORITHMS = {
    "score_kgmon": score_kgmon,
    "score_full_probmul_3": score_full_probmul_3,
}


def _decoded_basekeys(output_dir):
    p = Path(output_dir)
    if not p.exists():
        return []
    return sorted({x.name.split(".")[0] for x in p.iterdir() if x.is_file()})


data = ArcDataset.from_file(TEST_PATH)
if EFFECTIVE_MODE == "validation" and SOLUTION_PATH is not None:
    data = data.load_replies(SOLUTION_PATH)

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
if Path(OUTPUT_DIR).exists() and any(Path(OUTPUT_DIR).iterdir()):
    decoder.load_decoded_results(OUTPUT_DIR)

selection_algorithm = SELECTION_ALGORITHMS[SELECTION_ALGORITHM]
submission = data.get_submission(decoder.run_selection_algo(selection_algorithm) if decoder.decoded_results else None)

with open(SUBMISSION_PATH, "w") as f:
    json.dump(submission, f)

print("decoded_output_keys =", len(decoder.decoded_results))
print("decoded_basekeys =", _decoded_basekeys(OUTPUT_DIR)[:20])
print("submission_path =", SUBMISSION_PATH)
print("submission_tasks =", len(submission))
print("submission_exists =", Path(SUBMISSION_PATH).exists())

if EFFECTIVE_MODE == "validation" and SOLUTION_PATH is not None:
    if decoder.decoded_results:
        decoder.benchmark_selection_algos()
    with open(SUBMISSION_PATH) as f:
        reload_submission = json.load(f)
    print("validation_score =", data.validate_submission(reload_submission))
else:
    preview_keys = list(submission)[:5]
    print("preview_keys =", preview_keys)
